In [3]:
from tqdm import tqdm
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pandas.tseries.offsets import BDay
import bt

startDate = '2015-05-01'
endDate = '2023-06-01'
data = bt.get('ioo', start=startDate, end=endDate)

def calculate_rsi(data, window=14):
    if isinstance(data, pd.DataFrame):
        # Assuming 'close' is the relevant column in your DataFrame
        price_data = data.iloc[:,1]
    elif isinstance(data, pd.Series):
        price_data = data
    else:
        raise ValueError("Input must be a DataFrame or Series.")

    delta = price_data.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

rsi = calculate_rsi(data)

# Create the strategy

testStrategy = bt.Strategy('s1', [bt.algos.RunMonthly(),
                       bt.algos.SelectWhere(rsi < 20),
                       bt.algos.WeighEqually(),
                       bt.algos.Rebalance()])

s2 = bt.Strategy('s2', [bt.algos.RunMonthly(),
                       bt.algos.SelectAll(),
                       bt.algos.WeighEqually(),
                       bt.algos.Rebalance()])


# Create the backtest
test = bt.Backtest(testStrategy, data)
test2 = bt.Backtest(s2, data)

# Run the backtest
res = bt.run(test, test2)

# Plot the results
res.plot()

IndexError: single positional indexer is out-of-bounds